<a href="https://colab.research.google.com/github/550tealeaves/DATA-70500-working-with-data/blob/main/March_Madness_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# This week introduces JOINs — the most powerful concept in SQL. You'll combine tournament game data with player stats across two tables, build a prediction, and create your first matplotlib charts to visualize what the data tells you.

## Load the dataset

In [1]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# ── TOURNAMENT GAME LOG ─────────────────────────────────────────
games_data = [
  # team,       round,       opponent,    pts_for, pts_against, fg_pct, three_pct, reb, ast, to, margin
  ('Michigan',  'Round 64',  'Howard',     101, 80,  0.561, 0.441, 42, 18, 10, 21),
  ('Michigan',  'Round 32',  'Saint Louis', 95, 72,  0.531, 0.418, 40, 17, 11, 23),
  ('Michigan',  'Sweet 16',  'Alabama',     90, 77,  0.512, 0.385, 38, 15, 12, 13),
  ('Michigan',  'Elite 8',   'Tennessee',   95, 62,  0.571, 0.464, 44, 22,  8, 33),
  ('Michigan',  'Final Four','Arizona',     91, 73,  0.521, 0.444, 38, 22,  9, 18),
  ('UConn',     'Round 64',  'Furman',      90, 59,  0.544, 0.412, 45, 18, 11, 31),
  ('UConn',     'Round 32',  'UCLA',        73, 57,  0.482, 0.391, 38, 15, 10, 16),
  ('UConn',     'Sweet 16',  'Michigan St', 71, 52,  0.461, 0.378, 40, 16, 12, 19),
  ('UConn',     'Elite 8',   'Duke',        73, 72,  0.441, 0.217,  28, 12,  5,  1),
  ('UConn',     'Final Four','Illinois',    71, 62,  0.432, 0.364,  35, 14, 11,  9),
]
games_cols = ['team','round','opponent','pts_for','pts_against',
              'fg_pct','three_pct','rebounds','assists','turnovers','margin']
df_games = pd.DataFrame(games_data, columns=games_cols)

# ── PLAYER STATS (season averages) ──────────────────────────────
players_data = [
  # team,       name,                  pos, ppg,  rpg, apg,  fg_pct, three_pct, tournament_ppg
  ('Michigan',  'Yaxel Lendeborg',      'F', 18.4, 7.1, 1.8,  0.541,  0.381,  22.3),
  ('Michigan',  'Aday Mara',            'C', 14.2, 7.8, 1.4,  0.571,  0.000,  18.6),
  ('Michigan',  'Morez Johnson Jr.',    'F', 13.8, 6.9, 1.2,  0.558,  0.000,  14.2),
  ('Michigan',  'Elliot Cadeau',        'G', 11.3, 3.1, 8.6,  0.441,  0.362,  10.8),
  ('Michigan',  'Roddy Gayle Jr.',      'G', 12.1, 2.8, 3.2,  0.431,  0.388,  11.4),
  ('UConn',     'Tarris Reed Jr.',      'C', 14.3, 8.8, 1.9,  0.621,  0.000,  21.8),
  ('UConn',     'Alex Karaban',         'F', 13.4, 5.2, 2.1,  0.451,  0.388,   9.4),
  ('UConn',     'Solo Ball',            'G', 13.0, 3.1, 4.8,  0.421,  0.361,  12.2),
  ('UConn',     'Braylon Mullins',      'G', 12.0, 2.8, 2.4,  0.441,  0.398,  14.6),
  ('UConn',     'Silas Demary Jr.',     'G', 10.4, 2.9, 5.1,  0.411,  0.352,   9.8),
]
players_cols = ['team','name','pos','ppg','rpg','apg','fg_pct','three_pct','tournament_ppg']
df_players = pd.DataFrame(players_data, columns=players_cols)

# ── LOAD INTO SQLITE ────────────────────────────────────────────
conn = sqlite3.connect(':memory:')
df_games.to_sql('tournament_games', conn, index=False, if_exists='replace')
df_players.to_sql('players', conn, index=False, if_exists='replace')
print('Ready! Both tables loaded.')

Ready! Both tables loaded.


## Download tables for CSV

In [6]:
# Convert to CSV & download
df_players.to_csv('players_mm.csv', index=False)
df_games.to_csv('games_mm.csv', index=False)
from google.colab import files
files.download('players_mm.csv')
files.download('games_mm.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## New skills

| Skill | Function | Example |
| :--- | :--- | :--- |
| INNER JOIN | Returns rows that match in both tables | Players who have game log data in both tables |
| LEFT JOIN | Returns ALL rows from left each table, NULLs if no match on right | All players, even if their team has no game data |
| ON | Defines shared key connecting 2 tables | JOIN players ON players.team = games.team |
| JOIN + GROUP BY | Combines table, summarizes results | Avg player PPG per team, alongside avg game margin |
| matplotlib | Python library to create charts | Bar charts of pts scored per round, line charts of margins |

In [2]:
# Basic INNER JOIN — connect players to their team's avg tournament margin
result = pd.read_sql_query('''
    SELECT
        p.name,
        p.team,
        p.ppg,
        ROUND(AVG(g.margin), 1) AS team_avg_margin
    FROM players p
    INNER JOIN tournament_games g ON p.team = g.team
    GROUP BY p.name, p.team, p.ppg
    ORDER BY p.team, p.ppg DESC
''', conn)
print(result)

                name      team   ppg  team_avg_margin
0    Yaxel Lendeborg  Michigan  18.4             21.6
1          Aday Mara  Michigan  14.2             21.6
2  Morez Johnson Jr.  Michigan  13.8             21.6
3    Roddy Gayle Jr.  Michigan  12.1             21.6
4      Elliot Cadeau  Michigan  11.3             21.6
5    Tarris Reed Jr.     UConn  14.3             15.2
6       Alex Karaban     UConn  13.4             15.2
7          Solo Ball     UConn  13.0             15.2
8    Braylon Mullins     UConn  12.0             15.2
9   Silas Demary Jr.     UConn  10.4             15.2


## Task 1 - SQL + INNER JOIN
- JOIN the players table to the tournament_games table on team name. Show each player's name, their season PPG, and their team's average margin of victory in the tournament. Sort by team, then PPG descending.


## Task 2 - SQL + JOIN + GROUP BY
- For each team, calculate: the average season PPG of their roster, the average tournament PPG of their roster, and their average tournament margin. How much do players elevate their game in the tournament? Which team's roster shows a bigger jump from season to tournament averages?

## Task 3 - SQL + LEFT JOIN
- Run the same JOIN from Task 1 but as a LEFT JOIN instead of INNER JOIN. Does the result change? Explain in a comment why or why not — and describe a scenario where a LEFT JOIN would return different results than an INNER JOIN.

## Task 4 - SQL + JOIN + WHERE
- Use a JOIN to find all players whose tournament PPG is higher than their season PPG AND whose team won their toughest game (closest margin). For Michigan that's Alabama (13 pts). For UConn that's Duke (1 pt). Who stepped up when it mattered most?

## Task 5 - SQL + JOIN + HAVING
- JOIN the two tables and GROUP BY team. Use HAVING to show only the team whose average roster FG% is above 0.49. What does shooting efficiency tell us about tonight's matchup?

## Task 6 - matplotlib - chart
- Create TWO charts in the same Colab cell:  Chart 1 — Grouped bar chart: Points scored per round for Michigan (maize bars) and UConn (navy bars). Rounds on the x-axis, points on the y-axis.  Chart 2 — Line chart: Margin of victory by round for both teams on the same plot. Use plt.plot() with different colors and add a legend. What story do the two lines tell about each team's tournament journey?

In [ ]:
# CHART 1
import numpy as np

rounds  = ['Round 64','Round 32','Sweet 16','Elite 8','Final Four']
mich_pts = list(df_games[df_games['team']=='Michigan']['pts_for'])
uconn_pts= list(df_games[df_games['team']=='UConn']['pts_for'])

x = np.arange(len(rounds))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, mich_pts, width, label='Michigan', color='#FFCB05', edgecolor='#00274C')
bars2 = ax.bar(x + width/2, uconn_pts, width, label='UConn',    color='#002868', edgecolor='white')

ax.set_xlabel('Tournament Round')
ax.set_ylabel('Points Scored')
ax.set_title('Michigan vs UConn — Points Scored Per Round (2026 NCAA Tournament)')
ax.set_xticks(x)
ax.set_xticklabels(rounds, rotation=15)
ax.legend()
ax.bar_label(bars1, padding=3)
ax.bar_label(bars2, padding=3)
plt.tight_layout()
plt.show()


## Task 7 - Predict & reflect
- Michigan has scored 90+ in every game. UConn is 6-0 all-time in title games. Michigan's Lendeborg has a knee/ankle injury. UConn's Solo Ball may be limited by a foot injury. Write your data-backed prediction: who wins tonight and why? Cite at least 3 numbers from your queries or the charts.